# SynthPanel Tuning with SynthBench

This notebook shows how to use [SynthPanel](https://github.com/DataViking-Tech/synthpanel)
to generate synthetic survey responses and score them with SynthBench.

**Why this exists**: SynthPanel is a tool for creating synthetic survey
respondents using LLMs with persona conditioning. SynthBench measures how
good those respondents are. Together, they let you iterate on quality
without needing the MCP integration.

**What you'll do**:
1. Create a simple survey instrument (YAML)
2. Create persona definitions (YAML)
3. Run SynthPanel to generate responses
4. View raw output
5. Score the output with SynthBench
6. Try different temperatures and compare scores

## 1. Install

You need both synthpanel (response generation) and synthbench (scoring).

In [ ]:
!pip install synthpanel synthbench --quiet

## 2. Create a Simple Instrument

An instrument defines the survey questions. Here we create a minimal one
inline as YAML.

In [ ]:
instrument_yaml = """
name: tech_attitudes
description: Attitudes toward technology and AI
questions:
  - key: tech_helpful
    text: "Overall, do you think technology has made life better or worse for most people?"
    options:
      - "Much better"
      - "Somewhat better"
      - "Not much difference"
      - "Somewhat worse"
      - "Much worse"

  - key: ai_jobs
    text: "How concerned are you that AI will replace jobs in your field?"
    options:
      - "Very concerned"
      - "Somewhat concerned"
      - "Not too concerned"
      - "Not at all concerned"

  - key: ai_trust
    text: "How much do you trust AI systems to make fair decisions?"
    options:
      - "A great deal"
      - "A fair amount"
      - "Not too much"
      - "Not at all"
"""

# Save to a temp file
from pathlib import Path

Path("instrument.yaml").write_text(instrument_yaml)
print("Instrument saved to instrument.yaml")

## 3. Create Persona Definitions

Personas tell the LLM which demographic to role-play. The quality of
responses depends heavily on how personas are defined.

In [ ]:
personas_yaml = """
personas:
  - name: young_progressive
    demographics:
      age: "18-29"
      ideology: "Liberal"
      education: "College grad"
    biography: >-
      A 24-year-old college graduate who works in tech and follows
      progressive politics closely.

  - name: older_conservative
    demographics:
      age: "65+"
      ideology: "Conservative"
      education: "HS grad"
    biography: >-
      A 70-year-old retired factory worker from a rural area who
      watches cable news and votes Republican.

  - name: moderate_suburban
    demographics:
      age: "30-49"
      ideology: "Moderate"
      education: "Some college"
    biography: >-
      A 38-year-old suburban parent who works in healthcare and
      considers themselves politically moderate.
"""

Path("personas.yaml").write_text(personas_yaml)
print("Personas saved to personas.yaml")

## 4. Run SynthPanel and View Raw Output

SynthPanel generates responses for each persona across all questions.

> **Note**: This step requires an LLM API key (e.g., `ANTHROPIC_API_KEY`
> or `OPENAI_API_KEY`). If you don't have one, skip to step 5 to see
> what scored output looks like.

In [ ]:
import os

# Uncomment and set your key:
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

if os.environ.get("ANTHROPIC_API_KEY") or os.environ.get("OPENAI_API_KEY"):
    # Run synthpanel via CLI (sb-adapt wraps it for SynthBench)
    !synthpanel run \
        --instrument instrument.yaml \
        --personas personas.yaml \
        --samples 10 \
        --output ./panel_output
else:
    print("Skipped — set ANTHROPIC_API_KEY or OPENAI_API_KEY to run SynthPanel.")
    print("\nWithout an API key, you can still explore the SynthBench scoring")
    print(
        "workflow using the random or majority baselines (see getting_started.ipynb)."
    )

In [ ]:
# View raw output (if generated)
import json

output_dir = Path("./panel_output")
if output_dir.exists():
    output_files = sorted(output_dir.glob("*.json"))
    if output_files:
        with open(output_files[0]) as f:
            panel_data = json.load(f)
        print(json.dumps(panel_data, indent=2)[:2000])
        print("\n... (truncated)")
    else:
        print("No output files found.")
else:
    print("Panel output directory not found — run step 4 first.")

## 5. Score the Output with SynthBench

The `synthpanel` provider in SynthBench uses the SynthPanel CLI adapter
to score panel output against real human data from OpinionsQA.

If you don't have panel output from step 4, this cell uses the random
baseline as a stand-in to demonstrate the scoring workflow.

In [ ]:
output_dir = Path("./panel_output")

if output_dir.exists() and list(output_dir.glob("*.json")):
    # Score real panel output
    !synthbench run --provider synthpanel --suite smoke --samples 10 --output ./scored_results
else:
    # Fallback: use random baseline to show the scoring workflow
    print("No panel output — using random baseline to demonstrate scoring.")
    !synthbench run --provider random --suite smoke --samples 10 --output ./scored_results

## 6. Try Different Temperatures

Temperature controls how "random" the LLM's responses are. Lower
temperature = more deterministic, higher = more varied.

For synthetic survey respondents, the right temperature balances:
- **Too low**: responses are too concentrated (low entropy)
- **Too high**: responses become noise (high entropy)

Run the same benchmark at different temperatures and compare SPS scores
to find the sweet spot.

In [ ]:
# This cell demonstrates the comparison workflow.
# With a real provider, you'd vary the temperature parameter.
#
# Example with OpenRouter at different temperatures:
#
# for temp in [0.3, 0.7, 1.0]:
#     !synthbench run \
#         --provider openrouter \
#         --model anthropic/claude-haiku-4-5-20251001 \
#         --suite smoke \
#         --samples 30 \
#         --output ./temp_results_{temp}
#
# Then compare:
# !synthbench compare ./temp_results_0.3/*.json ./temp_results_1.0/*.json

print("Temperature tuning workflow:")
print("  1. Run benchmarks at different temperatures")
print("  2. Compare results with 'synthbench compare'")
print("  3. The paired bootstrap test tells you if differences are significant")
print("  4. Pick the temperature that maximizes SPS")

## 7. Compare Scores

After running at multiple settings, use `synthbench compare` to see which
configuration produced the best scores — with statistical significance.

In [ ]:
# List all scored results
scored_dir = Path("./scored_results")
if scored_dir.exists():
    for f in sorted(scored_dir.glob("*.json")):
        with open(f) as fh:
            d = json.load(fh)
        provider = d.get("config", {}).get("provider", "unknown")
        sps = d.get("scores", {}).get("sps", 0)
        print(f"  {provider:25s}  SPS={sps:.4f}  ({f.name})")
else:
    print("No scored results yet — run the cells above first.")

## Next Steps

- See [getting_started.ipynb](./getting_started.ipynb) for the full
  SynthBench walkthrough (topic analysis, HttpProvider, etc.).
- Read [METHODOLOGY.md](../METHODOLOGY.md) for the scoring framework.
- Explore the SynthPanel docs for advanced persona conditioning techniques.